# More seeds: Experiment F to 10 seeds, Experiment E to 10 draws, blind transfer for every student

Reviewer comments R2-1, R2-4 and R2 minor 4. Same code as `experiment_F_robustness.ipynb` and Experiment E; nothing
already published is overwritten.

* **F (Tables 16–17):** S0 / S1 / S2 for seeds **3–9** (seeds 0–2 already exist) → `experiment_F/experiment_F_seeds3_9.json`
  and `experiment_F_scores_seeds3_9.npz`, plus a 10-seed summary.
* **E (Table 11, IV-M):** labels-only vs probe + distillation for draws **5–9** → `E_scores_rep5_9.npz`, and the
  10-draw statistics (paired t, Wilcoxon, Pitman–Morgan) → `E_distillation_10rep.json`.
* **Blind transfer (Table 17):** every S0/S1/S2 student, seeds 0–9, on External datasets 1 and 2 →
  `experiment_F/external_blind_transfer_10seeds.json`.

**Resumable.** Each student is saved as soon as it is trained and skipped on a re-run, so a Colab disconnect costs at
most one student. Runtime: about 21 F students + 10 E students at 5–7 min each, so 3–3.5 h of GPU; run it across two
sessions if needed (just run all cells again).

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'                              # one folder per class
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
ESCORES    = f'{RES}/E_scores_5rep.npz'
DEPLOY     = f'{RES}/deploy'
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
REPS       = []           # Experiment F trains its own students (below)
EXPORT_REP = 1            # which trained student to export (1 = draw closest to the 5-seed mean)
EPOCHS, BS, LR = 30, 64, 1e-3

# ---- Experiment F ----
FIELD_ZIP_OR_DIR = f'{DRIVE}/Tea Field Dataset'
FIELD_FEAT_WHOLE = f'{RES}/field_test/feat_Field_whole.npz'   # from field_test.ipynb (alignment + split groups)
OUTF       = f'{RES}/experiment_F'
F_REPS     = [3, 4, 5, 6, 7, 8, 9]   # NEW seed draws (0-2 are published)
PI_SENS    = []          # S1 retrained with the field healthy prevalence misestimated by -/+10 points
PI_REP     = 1            # seed draw used for the prevalence-sensitivity runs
CENTRE_FRAC= 0.60         # centre crop = central 60% square of the frame (phone-feasible framing guide)
DUP        = 0.95         # near-duplicate grouping for the split and the bootstrap
FIELD_PI   = 0.26         # healthy prevalence of the field set (label-free rule; a field walk would estimate it)
TARGET_SENS= 0.95
TRAIN_S0, TRAIN_S2 = True, True

# ---- added for this notebook ----
E_REPS     = [5, 6, 7, 8, 9]       # NEW Experiment E draws (0-4 are published in E_scores_5rep.npz)
EXT_OUT    = f'{RES}/external_test'
EWU_DIR    = f'{DRIVE}/East West University dataset'
ALLY_DIR   = f'{DRIVE}/Allynugger'
F_JSON     = f'{OUTF}/experiment_F_seeds3_9.json'
F_NPZ      = f'{OUTF}/experiment_F_scores_seeds3_9.npz'


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
!pip -q install onnx onnxruntime timm
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(DEPLOY, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


In [ ]:
# CELL 5 — crop cache (JPEG q95, same as the revision_v1 run), parallel
def cache_path(rec):
    return os.path.join(CACHE, hashlib.md5(rec['rel'].encode()).hexdigest() + '.jpg')
def _build(rec):
    cp = cache_path(rec)
    if not os.path.exists(cp):
        try: leaf_crop(rec['path']).save(cp, quality=95)
        except Exception as e: return rec['rel'], str(e)
    return None
t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    fails = [f for f in ex.map(_build, RECS, chunksize=32) if f]
print('crop cache: %d images, %d failures, %.0fs' % (len(RECS), len(fails), time.time() - t0))
assert not fails, fails[:5]


In [ ]:
# Meta's server currently fails certificate verification in Colab; integrity is checked below (features must
# reproduce the cached training features). Normal verification is restored afterwards.
import ssl; _ssl_default = ssl._create_default_https_context; ssl._create_default_https_context = ssl._create_unverified_context
# CELL 6 — DINO ViT-S/16 features: use the cache only if our crops reproduce it
TO_T = T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
dino = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device).eval()
@torch.no_grad()
def feats(recs, bs=128):
    out = []
    for i in range(0, len(recs), bs):
        x = torch.stack([TO_T(Image.open(cache_path(r)).convert('RGB')) for r in recs[i:i + bs]]).to(device)
        out.append(torch.nn.functional.normalize(dino(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out)
rng = np.random.default_rng(0)
it, ie = rng.choice(len(TRAIN), 64, replace=False), rng.choice(len(TEST), 64, replace=False)
ct = (feats([TRAIN[i] for i in it]) * F['Xtr'][it]).sum(1) if TRAIN_ORDER_OK else np.zeros(1)
ce = (feats([TEST[i] for i in ie]) * F['Xte'][ie]).sum(1)
print('cosine to cache: train median %.4f | test median %.4f' % (np.median(ct), np.median(ce)))
USE_CACHE = TRAIN_ORDER_OK and np.median(ct) >= 0.99 and np.median(ce) >= 0.99
if USE_CACHE:
    Xtr, Xte = F['Xtr'], F['Xte']; print('-> using cached features (exact reproduction of the published run)')
else:
    print('-> cache does not align with these crops: re-extracting all features (numbers will differ slightly)')
    Xtr, Xte = feats(TRAIN), feats(TEST)
    np.savez(f'{RES}/features_dinov1_crop_rebuilt.npz', Xtr=Xtr, ytr=ytr, dtr=dtr, Xte=Xte, yte=yte, dte=dte)
ssl._create_default_https_context = _ssl_default   # keep DINO loaded for the field centre crops


In [ ]:
# CELL 7 — 70-label probe (multinomial logistic, C=1.0) and the binary score; reproduce published AUCs
J = json.load(open(EJSON))
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def fit_probe(X, y, idx):
    return LogisticRegression(C=1.0, max_iter=2000).fit(X[idx], y[idx])
def binary_score(probe, X):
    return 1.0 - probe.predict_proba(X)[:, HEALTHY]
pa = [roc_auc_score(dte, binary_score(fit_probe(Xtr, ytr, draw(r)), Xte)) for r in range(5)]
print('probe AUC per draw:', np.round(pa, 4)); print('published         :', np.round(J['probe_auc_per_rep'], 4))
if USE_CACHE:
    assert np.allclose(pa, J['probe_auc_per_rep'], atol=1e-3), 'probe did not reproduce: stop'
    print('-> exact reproduction')


In [ ]:
# CELL 8 — train the student(s): prevalence-quantile hard pseudo-labels, MobileNetV3-Small (Sec III-F)
class ArrDS(Dataset):
    def __init__(self, recs, labels, train):
        self.r, self.l, self.train = recs, labels, train
        self.aug = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                              T.RandomRotation(20), T.ColorJitter(.2, .2, .2)])
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        im = Image.open(cache_path(self.r[i])).convert('RGB')
        if self.train: im = self.aug(im)
        return TO_T(im), int(self.l[i])

def train_student(recs, labels, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2); m = m.to(device)
    dl = DataLoader(ArrDS(recs, labels, True), batch_size=BS, shuffle=True, num_workers=4)
    opt = optim.Adam(m.parameters(), lr=LR); sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    lf = nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        m.train()
        for x, y in dl:
            opt.zero_grad(); lf(m(x.to(device)), y.to(device)).backward(); opt.step()
        sch.step()
    return m.eval()

@torch.no_grad()
def student_scores(m, recs):
    dl = DataLoader(ArrDS(recs, np.zeros(len(recs)), False), batch_size=128, num_workers=4)
    return np.concatenate([torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy() for x, _ in dl])

E5 = np.load(ESCORES)['probe_distil']; pub = [roc_auc_score(dte, e) for e in E5]
STUDENTS, LOG = {}, []
for rep in REPS:
    idx = draw(rep); probe = fit_probe(Xtr, ytr, idx); s_tr = binary_score(probe, Xtr)
    pi = 1.0 - float(dtr.mean()); tau = float(np.quantile(s_tr, pi))
    pseudo = (s_tr > tau).astype(int); pseudo[idx] = dtr[idx]
    t0 = time.time(); m = train_student(TRAIN, pseudo, seed=1000 + rep); s = student_scores(m, TEST)
    a = roc_auc_score(dte, s); rho = spearmanr(s, E5[rep])[0]
    torch.save(m.state_dict(), f'{DEPLOY}/student_rep{rep}.pt'); np.save(f'{DEPLOY}/student_scores_rep{rep}.npy', s)
    STUDENTS[rep] = m
    LOG.append(dict(rep=rep, tau=tau, pseudo_acc=float((pseudo == dtr).mean()), auc=float(a),
                    published_auc=float(pub[rep]), spearman_vs_published=float(rho), minutes=(time.time() - t0) / 60))
    print('rep %d | tau %.4f | pseudo acc %.4f | AUC %.4f (published %.4f) | Spearman %.3f | %.1f min'
          % (rep, tau, LOG[-1]['pseudo_acc'], a, pub[rep], rho, LOG[-1]['minutes']))
lo, hi = min(pub), max(pub)
for L in LOG:
    L['within_published_spread'] = bool(lo - 0.005 <= L['auc'] <= hi + 0.005)
    if not L['within_published_spread']: print('WARNING rep %d outside the published spread %.4f-%.4f' % (L['rep'], lo, hi))
json.dump(LOG, open(f'{DEPLOY}/students.json', 'w'), indent=2)


In [ ]:
# CELL F1 — field records (same order as field_test.ipynb), whole-frame and centre crops into the cache
import zipfile
from collections import Counter
os.makedirs(OUTF, exist_ok=True)
FIELD_LOCAL = '/content/field_local'
if not os.path.exists(FIELD_LOCAL):
    zips = glob.glob(f'{FIELD_ZIP_OR_DIR}/**/*.zip', recursive=True)
    if zips:
        for z in zips: zipfile.ZipFile(z).extractall(FIELD_LOCAL)
    else: shutil.copytree(FIELD_ZIP_OR_DIR, FIELD_LOCAL)
    for z in glob.glob(f'{FIELD_LOCAL}/**/*.zip', recursive=True): zipfile.ZipFile(z).extractall(os.path.dirname(z))
def field_class(p):
    d = os.path.basename(os.path.dirname(p)).lower().replace('_', ' ')
    return ('Healthy' if 'healthy' in d else 'Red Rust' if 'rust' in d else 'Helopeltis' if 'helopelt' in d
            else 'Blight' if 'blight' in d else None)
files = sorted(p for p in glob.glob(f'{FIELD_LOCAL}/**/*', recursive=True)
               if p.lower().endswith(IMG_EXT) and '__MACOSX' not in p and field_class(p))
FREC = []
for p in files:
    c = field_class(p); rel = f'FIELD/{c}/{os.path.basename(p)}'
    FREC.append(dict(path=p, cls=c, d=int(c != 'Healthy'),
                     whole=dict(rel=rel + '#whole'), centre=dict(rel=rel + '#centre')))
fy = np.array([r['d'] for r in FREC]); fcls = np.array([r['cls'] for r in FREC])
W = np.load(FIELD_FEAT_WHOLE, allow_pickle=True)
assert len(fy) == len(W['y']) and (fy == W['y']).all() and (fcls == W['cls']).all(), \
    'field records do not align with feat_Field_whole.npz (same download/order as field_test.ipynb needed)'
Xf_whole = W['X']
print('field:', Counter(fcls), '| aligned with field_test features')
def _fcrops(r):
    rgb = np.array(Image.open(r['path']).convert('RGB')); H, Wd = rgb.shape[:2]
    out = {}
    for key, box in (('whole', (0, 0, Wd, H)),
                     ('centre', (Wd * (1 - CENTRE_FRAC) / 2, H * (1 - CENTRE_FRAC) / 2,
                                 Wd * (1 + CENTRE_FRAC) / 2, H * (1 + CENTRE_FRAC) / 2))):
        cp = cache_path(r[key])
        if not os.path.exists(cp):
            x0, y0, x1, y1 = map(int, box)
            Image.fromarray(rgb[y0:y1, x0:x1]).resize((IMG, IMG), Image.BILINEAR).save(cp, quality=95)
    return True
t0 = time.time()
with ThreadPoolExecutor(8) as ex: list(ex.map(_fcrops, FREC))
print('field crops cached (%.0fs)' % (time.time() - t0))


In [ ]:
# CELL F2 — centre-crop DINO features and zero-shot probe check (whole-frame features come from field_test)
Xf_centre = feats([r['centre'] for r in FREC])
del dino; torch.cuda.empty_cache()
print('centre crops: %d features' % len(Xf_centre))
ZS = {}
for view, X in (('whole', Xf_whole), ('centre', Xf_centre)):
    a = [roc_auc_score(fy, binary_score(fit_probe(Xtr, ytr, draw(r)), X)) for r in range(5)]
    ZS[view] = (float(np.mean(a)), float(np.std(a, ddof=1)))
    print('zero-shot probe, field %-6s AUC %.3f ± %.3f (5 draws)' % (view, *ZS[view]))


In [ ]:
# CELL F3 — field train/test split: near-duplicate groups kept together, stratified by class
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
grp = connected_components(csr_matrix(Xf_whole @ Xf_whole.T >= DUP), directed=False)[1]
rng = np.random.default_rng(2026)
is_tr = np.zeros(len(FREC), bool)
for c in np.unique(fcls):
    gs = np.unique(grp[fcls == c]); rng.shuffle(gs)
    cum, half = 0, (fcls == c).sum() / 2
    for g in gs:
        if cum >= half: break
        is_tr[grp == g] = True; cum += (grp == g).sum()
assert not set(grp[is_tr]) & set(grp[~is_tr]), 'a near-duplicate group crosses the split'
F_TR, F_TE = np.where(is_tr)[0], np.where(~is_tr)[0]
print('field-train %d | field-test %d | groups crossing the split: 0' % (len(F_TR), len(F_TE)))
print('field-test classes:', Counter(fcls[F_TE]))
json.dump(dict(dup=DUP, train=[FREC[i]['whole']['rel'][:-6] for i in F_TR], test=[FREC[i]['whole']['rel'][:-6] for i in F_TE]),
          open(f'{OUTF}/field_split.json', 'w'), indent=1)


In [ ]:
# CELL F4 — train S0 / S1 / S2 for every seed draw, plus S1 under misestimated prevalence; save all scores
from sklearn.metrics import roc_auc_score
def thr_at_sens(s, d, t=TARGET_SENS): return float(np.sort(s[d == 1])[int(np.floor((1 - t) * (d == 1).sum()))])
def sens_spec(s, d, thr): f = s >= thr; return float(f[d == 1].mean()), float((~f)[d == 0].mean())
def group_boot_auc(s, d, g, B=1000, seed=0):
    r = np.random.default_rng(seed); ug = np.unique(g); idx = {k: np.where(g == k)[0] for k in ug}; out = []
    for _ in range(B):
        sel = np.concatenate([idx[k] for k in r.choice(ug, len(ug), replace=True)])
        if len(np.unique(d[sel])) == 2: out.append(roc_auc_score(d[sel], s[sel]))
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]
FIELD_TR_RECS = [FREC[i][v] for i in F_TR for v in ('whole', 'centre')]
ct = np.unique(fcls); dF = fy[F_TE]
RES_F = json.load(open(F_JSON)) if os.path.exists(F_JSON) else {'zero_shot_probe': ZS, 'split': dict(n_train=int(len(F_TR)), n_test=int(len(F_TE)))}
SCORES, STU = ({k: v for k, v in np.load(F_NPZ).items()} if os.path.exists(F_NPZ) else
               {'tealeaf_test_d': dte, 'field_test_d': dF, 'field_test_cls': fcls[F_TE], 'field_test_group': grp[F_TE]}), {}

def run_student(name, rep, recs, labels, extra):
    if f'{name}_rep{rep}' in RES_F and os.path.exists(f'{OUTF}/student_{name}_rep{rep}.pt'):
        print('  %s rep %d already done -> skipped' % (name, rep)); return
    t0 = time.time(); m = train_student(recs, labels, seed=1000 + rep)
    s_te = student_scores(m, TEST); thr = thr_at_sens(s_te, dte)
    row = dict(rep=rep, minutes=round((time.time() - t0) / 60, 1), **extra,
               tealeaf_test=dict(auc=float(roc_auc_score(dte, s_te)), sens_spec_fixed=sens_spec(s_te, dte, thr)))
    SCORES[f'{name}_rep{rep}_tealeaf'] = s_te
    for view in ('whole', 'centre'):
        sv = student_scores(m, [FREC[i][view] for i in F_TE]); SCORES[f'{name}_rep{rep}_field_{view}'] = sv
        row[f'field_{view}'] = dict(auc=float(roc_auc_score(dF, sv)), auc_ci95_group=group_boot_auc(sv, dF, grp[F_TE]),
                                    sens_spec_fixed_tealeaf_thr=sens_spec(sv, dF, thr),
                                    sens_spec_label_free=sens_spec(sv, dF, float(np.quantile(sv, FIELD_PI))),
                                    flag_rate_by_class={c: float((sv[fcls[F_TE] == c] >= thr).mean()) for c in ct})
    torch.save(m.state_dict(), f'{OUTF}/student_{name}_rep{rep}.pt'); STU[(name, rep)] = m.cpu()
    RES_F[f'{name}_rep{rep}'] = row
    print('  %-10s rep %d: TeaLeaf %.3f | field whole %.3f [%.3f, %.3f] centre %.3f [%.3f, %.3f] | spec@fixed %.2f / %.2f | %.1f min'
          % (name, rep, row['tealeaf_test']['auc'], row['field_whole']['auc'], *row['field_whole']['auc_ci95_group'],
             row['field_centre']['auc'], *row['field_centre']['auc_ci95_group'],
             row['field_whole']['sens_spec_fixed_tealeaf_thr'][1], row['field_centre']['sens_spec_fixed_tealeaf_thr'][1], row['minutes']))
    torch.cuda.empty_cache(); json.dump(RES_F, open(F_JSON, 'w'), indent=2, default=float)
    np.savez(F_NPZ, **SCORES)      # saved after every student, so a crash loses nothing

def field_pl(probe, pi):
    sf = 0.5 * (binary_score(probe, Xf_whole[F_TR]) + binary_score(probe, Xf_centre[F_TR]))
    return (sf > np.quantile(sf, pi)).astype(int)

for rep in F_REPS:
    idx = draw(rep); probe = fit_probe(Xtr, ytr, idx); s_tr = binary_score(probe, Xtr)
    tau = float(np.quantile(s_tr, 1.0 - float(dtr.mean()))); pl_tea = (s_tr > tau).astype(int); pl_tea[idx] = dtr[idx]
    pl_s1 = field_pl(probe, FIELD_PI)
    r2 = np.random.default_rng(5000 + rep)
    lab = np.concatenate([r2.choice(F_TR[fcls[F_TR] == c], 10, replace=False) for c in ct])
    p2 = LogisticRegression(C=1.0, max_iter=2000).fit(np.vstack([Xf_whole[lab], Xf_centre[lab]]), np.concatenate([fcls[lab]] * 2))
    hk = list(p2.classes_).index('Healthy')
    s2 = 0.5 * ((1 - p2.predict_proba(Xf_whole[F_TR])[:, hk]) + (1 - p2.predict_proba(Xf_centre[F_TR])[:, hk]))
    pl_s2 = (s2 > np.quantile(s2, FIELD_PI)).astype(int); m_lab = np.isin(F_TR, lab); pl_s2[m_lab] = fy[F_TR[m_lab]]
    acc1, acc2 = float((pl_s1 == fy[F_TR]).mean()), float((pl_s2 == fy[F_TR]).mean())
    print('rep %d | field pseudo-label accuracy: S1 %.3f | S2 %.3f' % (rep, acc1, acc2))
    if TRAIN_S0: run_student('S0', rep, TRAIN, pl_tea, dict(expert_field_labels=0, field_pi=None, field_pseudo_acc=None))
    run_student('S1', rep, TRAIN + FIELD_TR_RECS, np.concatenate([pl_tea, np.repeat(pl_s1, 2)]),
                dict(expert_field_labels=0, field_pi=FIELD_PI, field_pseudo_acc=acc1))
    if TRAIN_S2: run_student('S2', rep, TRAIN + FIELD_TR_RECS, np.concatenate([pl_tea, np.repeat(pl_s2, 2)]),
                             dict(expert_field_labels=40, field_pi=FIELD_PI, field_pseudo_acc=acc2))
    if rep == PI_REP:
        for pi in PI_SENS:
            pl = field_pl(probe, pi); acc = float((pl == fy[F_TR]).mean())
            print('rep %d | S1 with field prevalence %.2f: pseudo-label accuracy %.3f' % (rep, pi, acc))
            run_student('S1_pi%02d' % round(100 * pi), rep, TRAIN + FIELD_TR_RECS, np.concatenate([pl_tea, np.repeat(pl, 2)]),
                        dict(expert_field_labels=0, field_pi=pi, field_pseudo_acc=acc))
print('saved ->', F_JSON, 'and', F_NPZ)


In [ ]:
# CELL F5' — 10-seed summary: published seeds 0-2 + new seeds 3-9
OLD = json.load(open(f'{OUTF}/experiment_F_robustness.json'))
ALL = {**{k: v for k, v in OLD.items() if re.match(r'S[012]_rep[0-2]$', k)}, **{k: v for k, v in RES_F.items() if re.match(r'S[012]_rep\d$', k)}}
SUM10 = {}
for k in ('S0', 'S1', 'S2'):
    reps = sorted(int(x.split('rep')[1]) for x in ALL if x.startswith(k + '_rep'))
    row = {'seeds': reps}
    for name, path in (('tealeaf_auc', ('tealeaf_test', 'auc')), ('field_whole_auc', ('field_whole', 'auc')), ('field_centre_auc', ('field_centre', 'auc'))):
        v = [ALL[f'{k}_rep{r}'][path[0]][path[1]] for r in reps]
        row[name] = [float(np.mean(v)), float(np.std(v, ddof=1)), float(np.min(v)), float(np.max(v))]
    for view in ('whole', 'centre'):
        v = np.array([ALL[f'{k}_rep{r}'][f'field_{view}']['sens_spec_fixed_tealeaf_thr'] for r in reps])
        row[f'field_{view}_sens_spec_fixed'] = [float(v[:, 0].mean()), float(v[:, 1].mean())]
    SUM10[k] = row
    print('%s (%d seeds): ours %.3f ± %.3f | field whole %.3f ± %.3f [%.3f, %.3f] | centre %.3f ± %.3f'
          % (k, len(reps), *row['tealeaf_auc'][:2], *row['field_whole_auc'], *row['field_centre_auc'][:2]))
common = sorted(set(SUM10['S0']['seeds']) & set(SUM10['S1']['seeds']))
d = np.array([ALL[f'S1_rep{r}']['field_whole']['auc'] - ALL[f'S0_rep{r}']['field_whole']['auc'] for r in common])
from scipy.stats import wilcoxon, ttest_rel
SUM10['paired_S1_minus_S0_whole'] = dict(per_seed=[float(x) for x in d], mean=float(d.mean()), wins=int((d > 0).sum()), n=len(d),
                                         wilcoxon_p=float(wilcoxon(d).pvalue))
print('S1 - S0 (field whole), %d seeds: mean %+.3f, S1 better on %d, Wilcoxon p = %.4f' % (len(d), d.mean(), (d > 0).sum(), wilcoxon(d).pvalue))
json.dump(SUM10, open(f'{OUTF}/experiment_F_summary_10seeds.json', 'w'), indent=1)


In [ ]:
# CELL E — Experiment E draws 5-9: labels-only vs probe + distillation (same recipe as Experiment E; per-draw torch seed)
from scipy.stats import ttest_rel, wilcoxon, pearsonr
E_NPZ = f'{RES}/E_scores_rep5_9.npz'
EN = {k: v for k, v in np.load(E_NPZ).items()} if os.path.exists(E_NPZ) else {}
for rep in E_REPS:
    if f'labels_only_rep{rep}' in EN and f'probe_distil_rep{rep}' in EN:
        print('E draw %d already done -> skipped' % rep); continue
    idx = draw(rep); seed_recs, seed_bin = [TRAIN[i] for i in idx], dtr[idx]
    t0 = time.time(); EN[f'labels_only_rep{rep}'] = student_scores(train_student(seed_recs, seed_bin, seed=2000 + rep), TEST)
    probe = fit_probe(Xtr, ytr, idx); s_tr = binary_score(probe, Xtr)
    tau = float(np.quantile(s_tr, 1.0 - float(dtr.mean()))); pseudo = (s_tr > tau).astype(int); pseudo[idx] = dtr[idx]
    EN[f'probe_distil_rep{rep}'] = student_scores(train_student(TRAIN, pseudo, seed=1000 + rep), TEST)
    np.savez(E_NPZ, **EN); torch.cuda.empty_cache()
    print('E draw %d: labels-only %.4f | probe+distil %.4f | %.1f min' % (rep, roc_auc_score(dte, EN[f'labels_only_rep{rep}']),
          roc_auc_score(dte, EN[f'probe_distil_rep{rep}']), (time.time() - t0) / 60))
E5 = np.load(ESCORES)
lo = [roc_auc_score(dte, s) for s in E5['labels_only']] + [roc_auc_score(dte, EN[f'labels_only_rep{r}']) for r in E_REPS]
pd_ = [roc_auc_score(dte, s) for s in E5['probe_distil']] + [roc_auc_score(dte, EN[f'probe_distil_rep{r}']) for r in E_REPS]
lo, pd_ = np.array(lo), np.array(pd_); diff = pd_ - lo
r_pm = pearsonr(pd_ + lo, pd_ - lo); n = len(lo)                     # Pitman-Morgan: equal variances <=> corr(sum, diff) = 0
t_pm = r_pm[0] * np.sqrt((n - 2) / (1 - r_pm[0] ** 2))
from scipy.stats import t as tdist
p_pm = float(2 * tdist.sf(abs(t_pm), n - 2))
E10 = dict(n_draws=n, labels_only=[float(lo.mean()), float(lo.std(ddof=1)), float(lo.min()), float(lo.max())],
           probe_distil=[float(pd_.mean()), float(pd_.std(ddof=1)), float(pd_.min()), float(pd_.max())],
           per_draw=dict(labels_only=[float(x) for x in lo], probe_distil=[float(x) for x in pd_]),
           paired_t_p=float(ttest_rel(pd_, lo).pvalue), wilcoxon_p=float(wilcoxon(diff).pvalue), wins=int((diff > 0).sum()),
           sd_ratio=float(lo.std(ddof=1) / pd_.std(ddof=1)), pitman_morgan_p=p_pm,
           note='draws 0-4 from E_scores_5rep.npz (published); draws 5-9 new, torch seed 2000+rep (labels-only) and 1000+rep (distil)')
json.dump(E10, open(f'{RES}/E_distillation_10rep.json', 'w'), indent=1)
print('10 draws: labels-only %.3f ± %.3f | probe+distil %.3f ± %.3f | paired t p=%.3f | Wilcoxon p=%.3f | SD ratio %.1f | Pitman-Morgan p=%.4f'
      % (*E10['labels_only'][:2], *E10['probe_distil'][:2], E10['paired_t_p'], E10['wilcoxon_p'], E10['sd_ratio'], p_pm))


In [ ]:
# CELL B1 — (verbatim from external_test.ipynb) preprocessing (same recipe as the paper: Otsu on HSV V|S, largest component,
#           fill holes, bbox crop with 6% padding, resize 224x224, JPEG q95 round-trip)
def leaf_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0)
    lab, n = ndimage.label(m)
    if n == 0:
        return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def box_of(mask):
    ys, xs = np.where(mask)
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1

def crop_box(rgb, box, pad=0.06, size=IMG):
    H, W = rgb.shape[:2]; x0, y0, x1, y1 = box
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0 = int(max(0, x0 - px)), int(max(0, y0 - py))
    x1, y1 = int(min(W, x1 + px)), int(min(H, y1 + py))
    im = Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=95)       # match the cached-crop pipeline
    return Image.open(io.BytesIO(buf.getvalue())).convert('RGB')

def load_rgb(p):
    return np.array(Image.open(p).convert('RGB'))

def otsu_crop(rgb):
    m = leaf_mask(rgb)
    return crop_box(rgb, box_of(m)), float(m.mean())

def whole_crop(rgb):
    H, W = rgb.shape[:2]
    return crop_box(rgb, (0, 0, W, H), pad=0.0), 1.0


In [ ]:
!pip -q install pyyaml
import yaml, zipfile
# CELL B2 — (verbatim from external_test.ipynb) EWU: unzip Roboflow export, read YOLO boxes, drop Roboflow augmented copies
EWU_X = '/content/ewu'
if not os.path.exists(EWU_X):
    z = [p for p in glob.glob(f'{EWU_DIR}/*.zip')]
    assert z, 'no zip found in ' + EWU_DIR
    zipfile.ZipFile(z[0]).extractall(EWU_X)
yml = glob.glob(f'{EWU_X}/**/data.yaml', recursive=True) or glob.glob(f'{EWU_DIR}/data.yaml')
names = yaml.safe_load(open(yml[0]))['names']
names = list(names.values()) if isinstance(names, dict) else list(names)
print('EWU classes:', names)

def rf_stem(fname):          # Roboflow: <orig>_jpg.rf.<hash>.jpg  -> <orig>
    m = re.match(r'^(.*?)_(jpe?g|png|bmp|webp)\.rf\.', os.path.basename(fname), re.I)
    return m.group(1) if m else os.path.splitext(os.path.basename(fname))[0]

def split_rank(p):           # prefer un-augmented splits when de-duplicating
    return 0 if '/test/' in p else 1 if '/valid/' in p else 2

imgs = sorted([p for p in glob.glob(f'{EWU_X}/**/*', recursive=True) if p.lower().endswith(IMG_EXT)],
              key=lambda p: (split_rank(p), p))   # deterministic: same copy kept every run
seen, EWU = set(), []
for p in imgs:
    st = rf_stem(p)
    if st in seen: continue
    lbl = re.sub(r'/images/', '/labels/', p); lbl = os.path.splitext(lbl)[0] + '.txt'
    if not os.path.exists(lbl): continue
    seen.add(st)
    for line in open(lbl).read().strip().splitlines():
        v = line.split()
        if len(v) != 5: continue          # skip polygon rows
        EWU.append(dict(path=p, cls=names[int(v[0])], yolo=tuple(map(float, v[1:]))))
print('EWU: %d unique source images, %d leaf boxes (of %d image files incl. augmentations)'
      % (len(seen), len(EWU), len(imgs)))
from collections import Counter; print(Counter(r['cls'] for r in EWU))

def yolo_to_box(yolo, W, H):
    cx, cy, w, h = yolo
    return (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H


In [ ]:
# CELL B3 — (verbatim from external_test.ipynb) Allynugger: one folder per class, copied to local disk first (Drive reads are slow)
ALLY_LOCAL = '/content/ally_local'
src_files = [p for p in sorted(glob.glob(f'{ALLY_DIR}/*/*')) if p.lower().endswith(IMG_EXT)]
def _cp(p):
    d = os.path.join(ALLY_LOCAL, *p.split('/')[-2:])
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(p, d)
    return d
t0 = time.time()
with ThreadPoolExecutor(16) as ex: local_files = list(ex.map(_cp, src_files))
ALLY = [dict(path=p, cls=os.path.basename(os.path.dirname(p))) for p in local_files]
print('Allynugger: %s  (copied in %.0fs)' % (Counter(r['cls'] for r in ALLY), time.time() - t0))


In [ ]:
# CELL B4 — blind transfer of every S0/S1/S2 student (seeds 0-9) to External datasets 1 and 2
def _one(args):
    r, mode = args
    try:
        rgb = load_rgb(r['path']); H, W = rgb.shape[:2]
        im = crop_box(rgb, yolo_to_box(r['yolo'], W, H)) if mode == 'box' else whole_crop(rgb)[0]
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=95); return buf.getvalue()
    except Exception: return None
def build(records, mode):
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(_one, [(r, mode) for r in records]))
    keep = [i for i, b in enumerate(res) if b is not None]
    return dict(crops=[Image.open(io.BytesIO(res[i])).convert('RGB') for i in keep],
                y=np.array([0 if 'healthy' in records[i]['cls'].lower() else 1 for i in keep]), cls=np.array([records[i]['cls'] for i in keep]))
XSETS = {'External_1': build(EWU, 'box'), 'External_2': build(ALLY, 'whole')}
for k, S in XSETS.items(): print(k, len(S['y']), 'healthy', (S['y'] == 0).sum())
assert len(XSETS['External_1']['y']) == 3007 and len(XSETS['External_2']['y']) == 883, 'external sets differ from Table 15'
def load_student(path):
    m = tvm.mobilenet_v3_small(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    m.load_state_dict(torch.load(path, map_location='cpu')); return m.to(device).eval()
@torch.no_grad()
def pil_scores(m, pils, bs=128):
    out = []
    for i in range(0, len(pils), bs):
        out.append(torch.softmax(m(torch.stack([TO_T(p) for p in pils[i:i + bs]]).to(device)), 1)[:, 1].cpu().numpy())
    return np.concatenate(out)
SC_OLD = np.load(f'{OUTF}/experiment_F_scores.npz'); SC_NEW = np.load(F_NPZ)
def tea_scores(name, rep): return (SC_OLD if rep <= 2 else SC_NEW)[f'{name}_rep{rep}_tealeaf']
BT = {}
for name in ('S0', 'S1', 'S2'):
    for rep in range(10):
        p = f'{OUTF}/student_{name}_rep{rep}.pt'
        if not os.path.exists(p): print('missing', p); continue
        m = load_student(p); thr = thr_at_sens(tea_scores(name, rep), dte)
        for tag, S in XSETS.items():
            s = pil_scores(m, S['crops']); f = s >= thr
            BT[f'{tag}/{name}/rep{rep}'] = dict(auc=float(roc_auc_score(S['y'], s)), sens=float(f[S['y'] == 1].mean()), spec=float((~f)[S['y'] == 0].mean()))
        del m; torch.cuda.empty_cache()
SUMB = {}
for tag in XSETS:
    SUMB[tag] = {}
    for name in ('S0', 'S1', 'S2'):
        reps = [r for r in range(10) if f'{tag}/{name}/rep{r}' in BT]
        v = {k: [BT[f'{tag}/{name}/rep{r}'][k] for r in reps] for k in ('auc', 'sens', 'spec')}
        SUMB[tag][name] = {k: [float(np.mean(x)), float(np.std(x, ddof=1)), float(np.min(x)), float(np.max(x))] for k, x in v.items()} | {'n': len(reps)}
    d = np.array([BT[f'{tag}/S1/rep{r}']['auc'] - BT[f'{tag}/S0/rep{r}']['auc'] for r in range(10) if f'{tag}/S1/rep{r}' in BT and f'{tag}/S0/rep{r}' in BT])
    SUMB[tag]['paired_S1_minus_S0_auc'] = dict(per_seed=[float(x) for x in d], mean=float(d.mean()), median=float(np.median(d)), wins=int((d > 0).sum()),
                                               n=len(d), wilcoxon_p=float(wilcoxon(d).pvalue))
    print('\n%s' % tag)
    for name in ('S0', 'S1', 'S2'):
        v = SUMB[tag][name]; print('  %s (%d seeds): AUC %.3f ± %.3f [%.3f, %.3f] | sens %.2f | spec %.2f' % (name, v['n'], *v['auc'], v['sens'][0], v['spec'][0]))
    q = SUMB[tag]['paired_S1_minus_S0_auc']; print('  S1 - S0: mean %+.3f, median %+.3f, S1 better on %d of %d, Wilcoxon p = %.4f' % (q['mean'], q['median'], q['wins'], q['n'], q['wilcoxon_p']))
json.dump(dict(per_model=BT, summary=SUMB), open(f'{OUTF}/external_blind_transfer_10seeds.json', 'w'), indent=1)
print('\nUpload: experiment_F_seeds3_9.json, experiment_F_scores_seeds3_9.npz, experiment_F_summary_10seeds.json,')
print('        external_blind_transfer_10seeds.json (experiment_F/), E_distillation_10rep.json, E_scores_rep5_9.npz (results_revision_v1/)')
